# 10. 메타분석 3 — 토픽 비중의 공변량 분석(STM 대체): 연도를 조정한 기대 편수와 로지스틱 회귀

- **목적**: 언어 집합이 2026년에 치우쳐 있으므로(한국어 25편 중 14편이 2026), 토픽별 "기대 편수"를 연도 구성에 맞춰 조정하고도 관측·기대 차이가 남는지 본다. 구조적 토픽 모델(STM)이 공변량으로 토픽 비중을 설명하는 것과 같은 질문을, 이미 배정된 LDA 토픽에 대해 회귀로 답한다. R의 stm이 설치되면 같은 입력으로 STM을 돌려 비교한다.
- **입력**: `lda_doc_topics.csv`(02의 문서 배정), `allc`(연도, 언어 집합)
- **계산**: (가) 연도 조정 기대 편수 $E_{adj}(t,l) = \sum_y \pi_{t,y} \cdot n_{l,y}$($\pi_{t,y}$ = 그 해 세계의 토픽 $t$ 비중, $n_{l,y}$ = 집합 $l$의 그 해 편수). (나) 토픽마다 로지스틱 회귀 $\mathrm{logit}\,P(\text{topic}=t) = \beta_0 + \beta_{ko} x_{ko} + \beta_{zh} x_{zh} + \beta_{ja} x_{ja} + \sum_y \beta_y \mathbb{1}[\text{year}=y]$, 전체 20,107편. 오즈비 $e^{\beta_l}$과 Wald 95% 신뢰구간. 집합 안 관측이 0편인 칸은 분리(separation)라 추정하지 않는다.
- **주의**: 토픽은 확률이 가장 큰 하나에 배정한 값이라 혼합 비중이 아니다. 신뢰구간은 작은 집합에서 넓다. 1위 토픽 benchmark·score는 2026년 문체 토픽이라 해석에서 뺀다(3.2절).


In [ ]:
import glob, json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"                                   # 입력: 전처리 산출(정제 CSV)
VER = ROOT / "verification"                                            # 수치의 정본 JSON
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)   # 출력: 노트북 그림. 원고 그림은 paper/figures/
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)    # 출력: 노트북 표
CACHE = ROOT / "analysis/output/cache"; CACHE.mkdir(parents=True, exist_ok=True) # 출력: 재실행용 캐시(git 제외)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
SUFFIX = "2023_2026q3"   # D30: 분석 기간 2023-01 ~ 2026-09 말뭉치

try:
    sys.path.insert(0, str(ROOT / "analysis/scripts"))
    import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

LANGS = ["ko", "zh", "ja"]
LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
YEARS = [2023, 2024, 2025, 2026]          # 2026은 1~9월(D30)
MIN_N = 10   # 투영 집합의 토픽별 건수가 이보다 작으면 lift를 보고하지 않는다

allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str})
cjk_signal = pd.read_csv(DATA / f"corpus_llm_agent_cjk_lang_{SUFFIX}.csv", dtype={"id": str})   # 키워드 신호 257편 + 사람 라벨(D36)
counts = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
YEAR_BASIS = counts["definitions"]["year_basis"]   # D31: "id" = arXiv id 접두(최초 제출 월)
LABEL_TARGET = "그 언어의 텍스트를 실제 연구 대상으로 다룸"; LABEL_MULTI = "다국어 실험의 한 언어로 포함"
cjk = cjk_signal[cjk_signal.cjk_lang].copy()       # 최종 대상 언어 집합 = 신호 ∧ 라벨 '실제 연구 대상'(D36)

# 정본 대조: 노트북이 읽은 자료가 정본 JSON과 같은지 확인한다
assert YEAR_BASIS == "id", "연도 기준이 D31(id 접두)이 아니다"
assert len(allc) == counts["pipeline"]["llm_agent_all"], "전체 건수가 정본과 다르다"
assert len(cjk_signal) == counts["pipeline"]["llm_agent_cjk_signal"], "대상 언어 신호 건수가 정본과 다르다"
assert len(cjk) == counts["pipeline"]["llm_agent_cjk_lang"], "대상 언어 집합 건수가 정본과 다르다"
assert (cjk_signal.lang_label != "").all(), "라벨 없는 신호 논문이 있다"
assert allc.groupby("year").size().to_dict() == {int(k): v for k, v in counts["llm_agent_by_year"].items()}
assert set(cjk.id) <= set(allc.id) and allc.id.is_unique
for l in LANGS:
    allc[f"lang_{l}"] = allc[f"lang_{l}"].astype(bool); allc[f"lang_{l}_signal"] = allc[f"lang_{l}_signal"].astype(bool)
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == counts["cjk_lang_by_lang_total"]
N_ALL_BY_YEAR = {int(k): v for k, v in counts["pipeline"]["unique_by_year"].items()}   # 분모: 다섯 cs 분류의 고유 논문 수
print("정본 대조 통과: LLM 에이전트", len(allc), "| 한중일어 신호", len(cjk_signal), "| 대상 언어 집합(사람 라벨)", len(cjk), "| 연도 기준", YEAR_BASIS, "| 기간", counts["definitions"]["period"][:24])

In [ ]:
# 메타분석 공통: spaCy 표제어 캐시(02에서 만든 것)와 LDA 문서 배정을 읽는다
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
DOMAIN_STOP = {"paper", "propose", "proposed", "method", "methods", "result", "results", "show", "shows", "model", "models", "based", "approach", "approaches",
               "task", "tasks", "performance", "llm", "llms", "agent", "agents", "large", "language", "framework", "novel", "existing", "demonstrate",
               "experiments", "experimental", "study", "work", "using", "use", "data", "code", "available", "https", "github", "com", "et", "al"}
STOP = set(ENGLISH_STOP_WORDS) | DOMAIN_STOP
lem = {}
for line in open(CACHE / f"lemmas_{SUFFIX}.jsonl"):
    r = json.loads(line); lem[r["id"]] = [t for t in r["t"] if t not in STOP and len(t) > 2 and not t.isdigit()]
assert set(lem) == set(allc.id), "표제어 캐시와 말뭉치의 id가 다르다"
dt = pd.read_csv(TAB / "lda_doc_topics.csv", dtype={"id": str}); assert len(dt) == len(allc)
allc = allc.merge(dt[["id", "lda_topic", "lda_prob"]], on="id", how="left")
ld = json.loads(Path(sorted(glob.glob(str(VER / "topics_lda_*.json")))[-1]).read_text()); K = ld["K"]
label = {int(t): v["label"] for t, v in ld["topics"].items()}
allc["ym"] = allc.year * 100 + allc.month
MONTHS = sorted(allc.ym.unique()); assert len(MONTHS) == 45 and MONTHS[0] == 202301 and MONTHS[-1] == 202609
print("문서", len(allc), "| 표제어 토큰", sum(len(v) for v in lem.values()), "| 토픽", K, "| 월", len(MONTHS))

## 1. 연도 조정 기대 편수 대 단순 기대 편수

In [ ]:
import statsmodels.api as sm
pi = pd.crosstab(allc.lda_topic, allc.year, normalize="columns")       # 토픽 × 연도 비중(열 합 1)
rows = []
for t in range(K):
    r = {"topic": t, "label": label[t], "world_n": int((allc.lda_topic == t).sum())}
    for l in LANGS:
        sub = allc[allc[f"lang_{l}"]]; n_ly = sub.groupby("year").size().reindex(YEARS).fillna(0)
        obs = int((sub.lda_topic == t).sum()); exp_naive = float((allc.lda_topic == t).mean() * len(sub)); exp_adj = float(sum(pi.loc[t, y] * n_ly[y] for y in YEARS if y in pi.columns))
        r.update({f"{l}_obs": obs, f"{l}_exp_naive": round(exp_naive, 2), f"{l}_exp_adj": round(exp_adj, 2), f"{l}_ratio_adj": round(obs / exp_adj, 2) if exp_adj > 0 else None})
    rows.append(r)
A = pd.DataFrame(rows).set_index("topic"); A.to_csv(TAB / "topic_lang_year_adjusted.csv")
chg = A[[c for c in A.columns if c.endswith("_exp_naive") or c.endswith("_exp_adj")]]
print("연도 조정으로 기대 편수가 1편 이상 움직인 칸:")
for l in LANGS:
    d = (A[f"{l}_exp_adj"] - A[f"{l}_exp_naive"]); big = d[d.abs() >= 1].sort_values()
    print(" ", LANG_NAME[l], ", ".join(f"{label[t]} {A.loc[t, f'{l}_exp_naive']:.1f}→{A.loc[t, f'{l}_exp_adj']:.1f}" for t in big.index))

## 2. 토픽별 로지스틱 회귀 — 언어 집합의 오즈비(연도 조정)

- 출력: `analysis/output/tables/topic_lang_logit.csv`, `analysis/output/figures/fig9c_topic_lang_or.png`
- 읽는 법: 오즈비 > 1이면 같은 해의 세계 논문보다 그 토픽에 들 확률이 높다. 신뢰구간이 1을 포함하지 않는 칸만 굵게 본다.


In [ ]:
X = pd.DataFrame({f"lang_{l}": allc[f"lang_{l}"].astype(float) for l in LANGS})
for y in YEARS[1:]: X[f"y{y}"] = (allc.year == y).astype(float)
X = sm.add_constant(X)
rows = []
for t in range(K):
    yv = (allc.lda_topic == t).astype(float)
    try:
        m = sm.Logit(yv, X).fit(disp=0, maxiter=200)
        for l in LANGS:
            obs = int(((allc.lda_topic == t) & allc[f"lang_{l}"]).sum())
            if obs == 0:
                rows.append({"topic": t, "label": label[t], "lang": l, "obs": 0, "OR": None, "ci_low": None, "ci_high": None, "p": None, "note": "obs 0 (separation)"}); continue
            b, se = m.params[f"lang_{l}"], m.bse[f"lang_{l}"]
            rows.append({"topic": t, "label": label[t], "lang": l, "obs": obs, "OR": round(float(np.exp(b)), 2), "ci_low": round(float(np.exp(b - 1.96 * se)), 2), "ci_high": round(float(np.exp(b + 1.96 * se)), 2), "p": round(float(m.pvalues[f"lang_{l}"]), 4), "note": ""})
    except Exception as e:
        for l in LANGS: rows.append({"topic": t, "label": label[t], "lang": l, "obs": None, "OR": None, "ci_low": None, "ci_high": None, "p": None, "note": f"fit failed: {e}"[:60]})
G = pd.DataFrame(rows)
from statsmodels.stats.multitest import multipletests
mask = G.p.notna(); G.loc[mask, "q_bh"] = multipletests(G.loc[mask, "p"].values, method="fdr_bh")[1].round(4)   # 다중 비교(47칸) BH 보정(리뷰 2026-10-14)
G.to_csv(TAB / "topic_lang_logit.csv", index=False)
sig = G[(G.OR.notna()) & ((G.ci_low > 1) | (G.ci_high < 1))].sort_values(["lang", "OR"], ascending=[True, False])
print("BH 보정 q<0.05인 칸:", ", ".join(f"{r.lang}:{r.label}({'+' if r.OR > 1 else '-'}, q={r.q_bh})" for r in G[G.q_bh < 0.05].itertuples()))
print("신뢰구간이 1을 포함하지 않는 칸(연도 조정 뒤):"); print(sig[["lang", "label", "obs", "OR", "ci_low", "ci_high", "p"]].to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(13, 6.5), sharey=True)
for ax, l in zip(axes, LANGS):
    g = G[(G.lang == l) & G.OR.notna()].sort_values("topic"); y = np.arange(len(g))
    ax.errorbar(g.OR, y, xerr=[g.OR - g.ci_low, g.ci_high - g.OR], fmt="o", color=["#C0392B" if (lo > 1 or hi < 1) else "#7F8C8D" for lo, hi in zip(g.ci_low, g.ci_high)][0] if False else "#2E86C1", ecolor="#B0B7C3", capsize=2, ms=4)
    for i, (lo, hi, orv) in enumerate(zip(g.ci_low, g.ci_high, g.OR)):
        if lo > 1 or hi < 1: ax.plot(orv, i, "o", color="#C0392B", ms=5)
    ax.axvline(1, color="k", lw=0.6); ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels([f"{t} {label[t]}" for t in g.topic], fontsize=7); ax.set_title(f"{LANG_NAME[l]} (n={int(allc[f'lang_{l}'].sum())})", fontsize=10); ax.set_xlabel("odds ratio (year-adjusted), log scale")
plt.suptitle("Topic membership odds ratios for target-language sets vs world, adjusted for year (red = 95% CI excludes 1)", fontsize=10); plt.tight_layout(); plt.savefig(FIG / "fig9c_topic_lang_or.png", dpi=160); plt.show()

## 3. STM(구조적 토픽 모델, R stm) 결과와의 비교

- 입력: `verification/stm_*.json`(16_stm.R: K=25, prevalence ~ 언어 집합 + 연도, 스펙트럴 초기화). 없으면 이 절은 건너뛴다.
- 계산: (가) STM 토픽을 LDA 토픽에 상위 10단어 자카드 유사도로 대응시킨다(일대일이 아니므로 가장 가까운 것). (나) STM의 언어 집합 효과(estimateEffect, 연도 조정)가 유의한(p<0.05) 토픽과 2절의 로지스틱 회귀가 유의한 토픽을 나란히 둔다. (다) 언어 집합의 평균 토픽 비중(theta) 대 세계 비율.
- 주의: STM은 문서를 토픽 혼합으로 보고(theta), 2절은 최대 확률 토픽 하나로 본다. 두 모델의 토픽 경계가 달라 대응은 참고용이다.


In [ ]:
stm_files = sorted(glob.glob(str(VER / "stm_2*.json")))
if stm_files:
    st = json.loads(Path(stm_files[-1]).read_text())
    assert st["n_docs"] == len(allc) and st["vocab_size"] == ld["vocab_size"], "STM 입력이 현재 LDA와 다르다"
    assert st.get("input", {}).get("set_sizes") == {f"lang_{l}": int(allc[f"lang_{l}"].sum()) for l in LANGS}, "STM 언어 집합이 이전 버전이다. 15·16을 재실행한다."
    import hashlib
    assert st["input"].get("corpus_sha256") == hashlib.sha256((DATA / f"corpus_llm_agent_all_{SUFFIX}.csv").read_bytes()).hexdigest(), "STM 말뭉치가 현재 입력과 다르다"
    print("STM:", st["r_version"], "stm", st["stm_version"], "| 수렴", st["converged"], "| EM 반복", st["em_iterations"], "|", st["elapsed_min"], "분")
    lda_words = {int(t): set(v["top_words"][:10]) for t, v in ld["topics"].items()}
    stm_topics = pd.DataFrame(st["topics"]); stm_topics["words"] = stm_topics.prob_words.str.split()
    def best_match(ws):
        ws = set(ws[:10]); sims = {t: len(ws & lw) / len(ws | lw) for t, lw in lda_words.items()}; t = max(sims, key=sims.get); return t, round(sims[t], 2)
    stm_topics[["lda_match", "jaccard"]] = pd.DataFrame([best_match(w) for w in stm_topics.words], index=stm_topics.index)
    stm_topics["lda_label"] = stm_topics.lda_match.map(label)
    eff = pd.DataFrame(st["effects_lang"]); eff["lang"] = eff.covariate.str.replace("lang_", "")
    tb = pd.DataFrame(st["theta_by_set"]).set_index("topic")
    stm_sig = eff[eff.p < 0.05].merge(stm_topics[["topic", "lda_match", "lda_label", "jaccard"]], on="topic")
    stm_sig["direction"] = np.where(stm_sig.estimate > 0, "+", "-")
    print("\nSTM에서 언어 집합 효과가 유의한(p<0.05) 토픽:")
    print(stm_sig.sort_values(["lang", "estimate"], ascending=[True, False])[["lang", "topic", "label", "lda_label", "jaccard", "estimate", "p", "direction"]].to_string(index=False))
    print("\n로지스틱 회귀(2절)에서 유의한 칸:", ", ".join(f"{LN}:{r.label}({'+' if r.OR > 1 else '-'})" for LN, r in [(r.lang, r) for r in sig.itertuples()]))
    stm_topics[["topic", "label", "lda_match", "lda_label", "jaccard", "mean_theta", "prob_words"]].to_csv(TAB / "stm_vs_lda_topics.csv", index=False)
    # 방향 일치: STM 유의 토픽을 LDA 대응 토픽으로 옮겨 2절의 유의 칸과 비교
    agree = []
    for r in stm_sig.itertuples():
        m = sig[(sig.lang == r.lang) & (sig.topic == r.lda_match)]
        agree.append({"lang": r.lang, "stm_topic": r.topic, "stm_label": r.label, "lda_label": r.lda_label, "jaccard": r.jaccard, "stm_dir": r.direction, "logit_dir": ("+" if m.OR.iloc[0] > 1 else "-") if len(m) else "n.s."})
    AG = pd.DataFrame(agree); print("\nSTM 유의 토픽의 LDA 대응과 로지스틱 회귀 방향:"); print(AG.to_string(index=False) if len(AG) else "(없음)")
    stm_summary = {"file": Path(stm_files[-1]).name, "converged": st["converged"], "em_iterations": st["em_iterations"], "elapsed_min": st["elapsed_min"], "significant": stm_sig.to_dict(orient="records"), "agreement": AG.to_dict(orient="records"), "topic_match": stm_topics[["topic", "label", "lda_match", "lda_label", "jaccard", "mean_theta"]].to_dict(orient="records")}
else:
    stm_summary = None; print("STM 정본이 아직 없다(16_stm.R 실행 전).")

## 4. 정본 저장

In [ ]:
out = {"generated": TODAY, "notebook": "analysis/notebooks/10_topic_covariates.ipynb", "method": "year-adjusted expected counts; per-topic logistic regression with language indicators and year dummies (statsmodels Logit, Wald 95% CI)", "stm": stm_summary,
       "year_adjusted": {str(t): {c: (None if pd.isna(v) else (float(v) if not isinstance(v, str) else v)) for c, v in r.items()} for t, r in A.iterrows()},
       "logit": G.to_dict(orient="records"), "significant": sig.to_dict(orient="records"), "excluded_topic_note": "benchmark·score(2026 style topic) excluded from interpretation"}
p = VER / f"meta_topic_covariates_{TODAY}.json"; p.write_text(json.dumps(out, ensure_ascii=False, indent=1, default=lambda x: None if pd.isna(x) else float(x))); print("saved", p)